# Colab 7
## Decision trees in spark

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

import pyspark
from pyspark.sql import *
from pyspark.sql.types import *
from pyspark.sql.functions import *
from pyspark import SparkContext, SparkConf

Let's first initialise the spark context

In [3]:
conf = SparkConf() \
    .set("spark.ui.port", "4050") \
    .set("spark.driver.cores", "4") \
    .set("spark.driver.memory", "16g") \
# create the context
sc = pyspark.SparkContext(conf=conf)
spark = SparkSession.builder.getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/02/23 23:39:05 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


### Mnist dataset
For this colab, we will use the famous MNIST dataset, a large collections of hand written digits, widely used for training and testing in machine learning 

In [6]:
training = spark.read.format("libsvm").load("../../assets/datasets/mnist-digits-train.txt")
test = spark.read.format("libsvm").load("../../assets/datasets/mnist-digits-test.txt")

# Cache data for multiple uses
training.cache()
test.cache()

26/02/23 23:41:52 WARN LibSVMFileFormat: 'numFeatures' option not specified, determining the number of features by going though the input. If you know the number in advance, please specify it via 'numFeatures' option to avoid the extra scan.
26/02/23 23:41:57 WARN LibSVMFileFormat: 'numFeatures' option not specified, determining the number of features by going though the input. If you know the number in advance, please specify it via 'numFeatures' option to avoid the extra scan.


DataFrame[label: double, features: vector]

In [7]:
training.printSchema()
test.printSchema()

root
 |-- label: double (nullable = true)
 |-- features: vector (nullable = true)

root
 |-- label: double (nullable = true)
 |-- features: vector (nullable = true)



In [9]:
print(test.count())
print(training.count())

10000
60000


We will now build a simple decision tree, with the use of the spark MLLIb

In [ ]:
from pyspark.ml import Pipeline
from pyspark.ml.classification import DecisionTreeClassifier
from pyspark.ml.feature import StringIndexer, VectorIndexer
from pyspark.ml.evaluation import MulticlassClassificationEvaluator

In [16]:
dt = DecisionTreeClassifier(labelCol="label", featuresCol="features")

pipeline = Pipeline(stages=[dt])

model = pipeline.fit(training)
predictions = model.transform(test)

In [19]:
predictions.show(5)

+-----+--------------------+--------------------+--------------------+----------+
|label|            features|       rawPrediction|         probability|prediction|
+-----+--------------------+--------------------+--------------------+----------+
|  7.0|(778,[202,203,204...|[177.0,72.0,298.0...|[0.03228748631886...|       7.0|
|  2.0|(778,[94,95,96,97...|[20.0,43.0,200.0,...|[0.03460207612456...|       6.0|
|  1.0|(778,[128,129,130...|[0.0,5198.0,90.0,...|[0.0,0.9183745583...|       1.0|
|  0.0|(778,[124,125,126...|[4322.0,2.0,64.0,...|[0.92607670880651...|       0.0|
|  4.0|(778,[150,151,159...|[70.0,0.0,103.0,9...|[0.05649717514124...|       9.0|
+-----+--------------------+--------------------+--------------------+----------+
only showing top 5 rows


In [20]:
evaluator = MulticlassClassificationEvaluator(
    labelCol="label", predictionCol="prediction", metricName="accuracy")
accuracy = evaluator.evaluate(predictions)
print("Test Error = %g " % (1.0 - accuracy))

Test Error = 0.3216 


Let's find the max depth of the decision tree and other statistics

In [30]:
treeModel = model.stages[0]
print(treeModel)

DecisionTreeClassificationModel: uid=DecisionTreeClassifier_0809ff555294, depth=5, numNodes=63, numClasses=10, numFeatures=780
